# GPU vs CPU 性能对比实验

这个 notebook 演示了「notebook 工作流」和「脚本工作流」的区别。

**对比：用 .py 脚本 vs 用 notebook 做同一个实验**

## 第 1 步：先试一个小矩阵，看看能不能跑

In [1]:
import torch
import time

print(f"CUDA 可用: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

CUDA 可用: True
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


✅ 跑通了。接下来换个单元格，试试 500×500 矩阵乘法。

In [5]:
size = 500
a = torch.randn(size, size)
b = torch.randn(size, size)

# CPU
start = time.time()
_ = a @ b
print(f"CPU 500x500: {time.time() - start:.4f}s")

# GPU
a_gpu = a.to("cuda")
b_gpu = b.to("cuda")
torch.cuda.synchronize()
start = time.time()
_ = a_gpu @ b_gpu
torch.cuda.synchronize()
print(f"GPU 500x500: {time.time() - start:.4f}s")

CPU 500x500: 0.0020s
GPU 500x500: 0.0020s


## 第 2 步：500 太小看不出差距，改大点试试

**这就是 notebook 的核心优势——改一个参数不用重跑全部，只跑这个单元格就行。**

In [ ]:
size = 4000  # 只改这一个数字，Shift+Enter 重新跑
a = torch.randn(size, size)
b = torch.randn(size, size)

start = time.time()
_ = a @ b
cpu_t = time.time() - start

a_gpu = a.to("cuda")
b_gpu = b.to("cuda")
torch.cuda.synchronize()
start = time.time()
_ = a_gpu @ b_gpu
torch.cuda.synchronize()
gpu_t = time.time() - start

print(f"CPU: {cpu_t:.3f}s")
print(f"GPU: {gpu_t:.3f}s")
print(f"加速: {cpu_t / gpu_t:.0f}x")

## 第 3 步：顺手画个图

和上面代码在同一个 kernel 里，所以 `size`、`cpu_t`、`gpu_t` 直接用。

In [ ]:
import matplotlib.pyplot as plt

sizes = [500, 1000, 2000, 4000]
cpu_times = []
gpu_times = []

for s in sizes:
    a = torch.randn(s, s)
    b = torch.randn(s, s)
    
    t0 = time.time()
    _ = a @ b
    cpu_times.append(time.time() - t0)
    
    a_g = a.to("cuda")
    b_g = b.to("cuda")
    torch.cuda.synchronize()
    t0 = time.time()
    _ = a_g @ b_g
    torch.cuda.synchronize()
    gpu_times.append(time.time() - t0)

plt.figure(figsize=(8, 4))
plt.plot(sizes, cpu_times, 'o-', label='CPU')
plt.plot(sizes, gpu_times, 'o-', label='GPU (RTX 3060)')
plt.xlabel('矩阵大小')
plt.ylabel('时间 (s)')
plt.title('CPU vs GPU 矩阵乘法性能')
plt.legend()
plt.grid(True)
plt.show()

---

## 总结：脚本 vs Notebook

| 操作 | .py 脚本 | .ipynb notebook |
|------|---------|----------------|
| 改一个参数看效果 | 改代码 → 重跑整个文件 | 改参数 → Shift+Enter 只跑当前格 |
| 看上次运行结果 | 翻终端历史 | 结果直接嵌在代码下面 |
| 分享给师兄/导师 | 发 .py + 解释哪里看输出 | 发 .ipynb，代码图表全在一起 |
| 跑一整晚训练 | ✅ 适合 | ❌ 不适合（浏览器关了 kernel 就挂了） |

**一句话：notebook 是草稿纸，.py 是定稿。**